# 🔬 crdt-merge v0.5.0 — A100 Feature Tests

Stress-testing **crdt-merge v0.5.0** new features on an NVIDIA A100 GPU runtime:
wire-format serialization, probabilistic data structures (HLL, Bloom, CMS),
and backward-compatibility with v0.4.0 features.

Copyright 2026 Ryan Gillespie — Apache-2.0

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
!pip install -q crdt-merge==0.5.0 psutil

import crdt_merge, sys, platform, psutil, torch

print(f'crdt-merge  {crdt_merge.__version__}')
assert crdt_merge.__version__.startswith('0.5'), f'Expected 0.5.x, got {crdt_merge.__version__}'
print(f'Python      {sys.version}')
print(f'Platform    {platform.platform()}')
print(f'CPU cores   {psutil.cpu_count(logical=True)}')
print(f'RAM         {psutil.virtual_memory().total / 1e9:.1f} GB')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU         {props.name}')
    print(f'VRAM        {props.total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No CUDA device detected — running CPU-only')

In [ ]:
# ── Utilities ──────────────────────────────────────────────────────────
import time, json, gc, tracemalloc
import pandas as pd, numpy as np, random

RESULTS = []

def record(suite, test, rows=None, trials=None, duration_ms=None,
           mem_mb=None, extra=None):
    entry = {'suite': suite, 'test': test}
    if rows is not None: entry['rows'] = rows
    if trials is not None: entry['trials'] = trials
    if duration_ms is not None: entry['duration_ms'] = round(duration_ms, 2)
    if mem_mb is not None: entry['mem_mb'] = round(mem_mb, 2)
    if extra is not None: entry.update(extra)
    RESULTS.append(entry)
    tag = f'{rows:>10,} rows' if rows else (f'{trials:>6,} trials' if trials else '')
    ms  = f'{duration_ms:>10,.1f} ms' if duration_ms else ''
    mb  = f'{mem_mb:>8,.1f} MB' if mem_mb else ''
    print(f'  ✔ {test:<40s} {tag}  {ms}  {mb}')

class MemTracker:
    def __enter__(self):
        gc.collect()
        tracemalloc.start()
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, *exc):
        self.duration_ms = (time.perf_counter() - self.t0) * 1000
        _, self.peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        self.peak_mb = self.peak / 1e6

print('Utilities loaded ✓')

## Suite 1: WIRE_FORMAT — Serialization Benchmarks

Roundtrip serialize/deserialize on all five CRDT types at progressive scales.
Verify peek_type returns snake_case strings and wire_size returns a dict.

In [ ]:
# ── Suite 1: Wire Format Benchmarks ───────────────────────────────────
from crdt_merge import wire
from crdt_merge import GCounter, PNCounter, LWWRegister, ORSet, LWWMap

# -- GCounter at progressive node counts --
for num_nodes in [100, 1_000, 10_000]:
    gc_obj = GCounter()
    for i in range(num_nodes):
        gc_obj.increment(f'node_{i}', random.randint(1, 100))
    original_value = gc_obj.value

    with MemTracker() as mt:
        data = wire.serialize(gc_obj)
        restored = wire.deserialize(data)
    assert restored.value == original_value, f'GCounter value mismatch: {restored.value} != {original_value}'

    pt = wire.peek_type(data)
    assert pt == 'g_counter', f'Expected g_counter, got {pt}'

    ws = wire.wire_size(data)
    total_bytes = ws['total_bytes']

    record('WIRE_FORMAT', f'GCounter({num_nodes:,} nodes)',
           duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
           extra={'bytes': total_bytes, 'peek_type': pt,
                  'header_bytes': ws['header_bytes'],
                  'payload_bytes': ws['payload_bytes']})
    gc.collect()

# -- PNCounter --
pn = PNCounter()
for i in range(1000):
    pn.increment(f'n_{i}', random.randint(1, 50))
    pn.decrement(f'n_{i}', random.randint(0, 10))
orig_pn = pn.value

with MemTracker() as mt:
    data = wire.serialize(pn)
    restored = wire.deserialize(data)
assert restored.value == orig_pn
assert wire.peek_type(data) == 'pn_counter'
ws = wire.wire_size(data)
record('WIRE_FORMAT', 'PNCounter(1K nodes)',
       duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
       extra={'bytes': ws['total_bytes'], 'peek_type': 'pn_counter'})

# -- LWWRegister --
reg = LWWRegister(value='A' * 10_000)
with MemTracker() as mt:
    data = wire.serialize(reg)
    restored = wire.deserialize(data)
assert restored.value == reg.value
assert wire.peek_type(data) == 'lww_register'
ws = wire.wire_size(data)
record('WIRE_FORMAT', 'LWWRegister(10KB string)',
       duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
       extra={'bytes': ws['total_bytes'], 'peek_type': 'lww_register'})

# -- ORSet --
for n_items in [100, 1_000, 10_000]:
    os_obj = ORSet()
    for i in range(n_items):
        os_obj.add(f'item_{i}')
    with MemTracker() as mt:
        data = wire.serialize(os_obj)
        restored = wire.deserialize(data)
    for i in range(min(10, n_items)):
        assert restored.contains(f'item_{i}'), f'ORSet missing item_{i}'
    assert wire.peek_type(data) == 'or_set'
    ws = wire.wire_size(data)
    record('WIRE_FORMAT', f'ORSet({n_items:,} items)',
           duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
           extra={'bytes': ws['total_bytes'], 'peek_type': 'or_set'})
    gc.collect()

# -- LWWMap --
for n_keys in [100, 1_000, 10_000]:
    lm = LWWMap()
    for i in range(n_keys):
        lm.set(f'key_{i}', f'val_{i}')
    with MemTracker() as mt:
        data = wire.serialize(lm)
        restored = wire.deserialize(data)
    assert restored.get('key_0') == 'val_0'
    assert wire.peek_type(data) == 'lww_map'
    ws = wire.wire_size(data)
    record('WIRE_FORMAT', f'LWWMap({n_keys:,} keys)',
           duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
           extra={'bytes': ws['total_bytes'], 'peek_type': 'lww_map'})
    gc.collect()

# -- Batch serialize/deserialize --
objects = [GCounter(initial=i) for i in range(100)]
with MemTracker() as mt:
    batch_data = wire.serialize_batch(objects)
    restored_list = wire.deserialize_batch(batch_data)
assert len(restored_list) == 100
record('WIRE_FORMAT', 'serialize_batch(100 GCounters)',
       duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
       extra={'count': 100})

print(f'\nSuite 1 complete ✓')

## Suite 2: WIRE_COMPRESSION — Compression Benchmarks

Compare uncompressed vs compressed serialization across CRDT types and scales.

In [ ]:
# ── Suite 2: Wire Compression ─────────────────────────────────────────
from crdt_merge import wire, GCounter, PNCounter, ORSet, LWWMap, LWWRegister

def make_test_objects():
    gc_obj = GCounter()
    for i in range(5000):
        gc_obj.increment(f'node_{i}', random.randint(1, 100))

    pn_obj = PNCounter()
    for i in range(5000):
        pn_obj.increment(f'n_{i}', random.randint(1, 50))
        pn_obj.decrement(f'n_{i}', random.randint(0, 10))

    os_obj = ORSet()
    for i in range(5000):
        os_obj.add(f'element_{i}')

    lm_obj = LWWMap()
    for i in range(5000):
        lm_obj.set(f'k_{i}', f'value_{i}_' + 'x' * 50)

    reg_obj = LWWRegister(value='data_' * 2000)

    return [
        ('GCounter_5K', gc_obj),
        ('PNCounter_5K', pn_obj),
        ('ORSet_5K', os_obj),
        ('LWWMap_5K', lm_obj),
        ('LWWRegister_10KB', reg_obj),
    ]

test_objects = make_test_objects()

for name, obj in test_objects:
    # Uncompressed
    t0 = time.perf_counter()
    raw = wire.serialize(obj, compress=False)
    t_raw = (time.perf_counter() - t0) * 1000
    raw_size = wire.wire_size(raw)['total_bytes']

    # Compressed
    t0 = time.perf_counter()
    comp = wire.serialize(obj, compress=True)
    t_comp = (time.perf_counter() - t0) * 1000
    comp_info = wire.wire_size(comp)
    comp_size = comp_info['total_bytes']

    ratio = comp_size / raw_size if raw_size > 0 else 1.0
    savings = (1 - ratio) * 100

    # Verify compressed roundtrip
    restored = wire.deserialize(comp)

    record('WIRE_COMPRESS', f'{name}',
           duration_ms=t_comp,
           extra={
               'raw_bytes': raw_size,
               'compressed_bytes': comp_size,
               'ratio': round(ratio, 4),
               'savings_pct': round(savings, 1),
               'raw_ms': round(t_raw, 2),
               'comp_ms': round(t_comp, 2),
               'compressed': comp_info.get('compressed', False),
           })
    gc.collect()

print(f'\nSuite 2 complete ✓')

## Suite 3: PROBABILISTIC_SCALE — HLL, Bloom, CMS at Scale

Benchmark probabilistic data structures at progressive item counts up to 5M.

In [ ]:
# ── Suite 3: Probabilistic at Scale ───────────────────────────────────
from crdt_merge import probabilistic

SCALES_PROB = [1_000, 10_000, 100_000, 1_000_000, 5_000_000]

# --- HyperLogLog ---
print('--- HyperLogLog ---')
for n in SCALES_PROB:
    hll = probabilistic.MergeableHLL(precision=14)
    items = [f'item_{i}' for i in range(n)]

    with MemTracker() as mt:
        hll.add_all(items)
        est = hll.cardinality()

    error_pct = abs(est - n) / n * 100
    record('PROB_HLL', f'HLL({n:,} items)',
           rows=n, duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
           extra={'estimate': est, 'actual': n,
                  'error_pct': round(error_pct, 2),
                  'size_bytes': hll.size_bytes})
    del items
    gc.collect()

# HLL merge test
hll_a = probabilistic.MergeableHLL(precision=14)
hll_b = probabilistic.MergeableHLL(precision=14)
hll_a.add_all([f'a_{i}' for i in range(50_000)])
hll_b.add_all([f'b_{i}' for i in range(50_000)])
with MemTracker() as mt:
    hll_merged = hll_a.merge(hll_b)
    merged_est = hll_merged.cardinality()
assert merged_est > 50_000, f'Merged HLL estimate too low: {merged_est}'
record('PROB_HLL', 'HLL merge(50K+50K)',
       duration_ms=mt.duration_ms, extra={'estimate': merged_est})
gc.collect()

# --- Bloom Filter ---
print('\n--- Bloom Filter ---')
for n in SCALES_PROB:
    bloom = probabilistic.MergeableBloom(capacity=n, fp_rate=0.01)
    items = [f'item_{i}' for i in range(n)]

    with MemTracker() as mt:
        bloom.add_all(items)

    # Check membership
    found = sum(1 for i in range(min(1000, n)) if bloom.contains(f'item_{i}'))
    assert found == min(1000, n), f'Bloom false negatives: {found}/{min(1000, n)}'

    # Check false positive rate
    fp_checks = 10000
    false_positives = sum(
        1 for i in range(fp_checks)
        if bloom.contains(f'NONEXISTENT_{i}')
    )
    fp_rate = false_positives / fp_checks

    record('PROB_BLOOM', f'Bloom({n:,} items)',
           rows=n, duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
           extra={'fp_rate': round(fp_rate, 4),
                  'size_bytes': bloom.size_bytes})
    del items
    gc.collect()

# Bloom merge test
bl_a = probabilistic.MergeableBloom(capacity=10_000, fp_rate=0.01)
bl_b = probabilistic.MergeableBloom(capacity=10_000, fp_rate=0.01)
bl_a.add_all([f'a_{i}' for i in range(5_000)])
bl_b.add_all([f'b_{i}' for i in range(5_000)])
with MemTracker() as mt:
    bl_merged = bl_a.merge(bl_b)
assert bl_merged.contains('a_0')
assert bl_merged.contains('b_0')
record('PROB_BLOOM', 'Bloom merge(5K+5K)',
       duration_ms=mt.duration_ms, extra={'size_bytes': bl_merged.size_bytes})
gc.collect()

# --- Count-Min Sketch ---
print('\n--- Count-Min Sketch ---')
for n in SCALES_PROB:
    cms = probabilistic.MergeableCMS(width=max(1000, n // 10), depth=5)
    items = [f'item_{i % (n // 10)}' for i in range(n)]

    with MemTracker() as mt:
        cms.add_all(items)

    # Check estimates with .estimate() — NOT .count()
    sample_item = 'item_0'
    est = cms.estimate(sample_item)

    record('PROB_CMS', f'CMS({n:,} items)',
           rows=n, duration_ms=mt.duration_ms, mem_mb=mt.peak_mb,
           extra={'sample_estimate': est, 'total': cms.total,
                  'size_bytes': cms.size_bytes})
    del items
    gc.collect()

# CMS merge test
cms_a = probabilistic.MergeableCMS(width=1000, depth=5)
cms_b = probabilistic.MergeableCMS(width=1000, depth=5)
for i in range(1000):
    cms_a.add(f'item_{i}', 1)
    cms_b.add(f'item_{i}', 2)
with MemTracker() as mt:
    cms_merged = cms_a.merge(cms_b)
merged_est = cms_merged.estimate('item_0')
assert merged_est >= 3, f'CMS merged estimate too low: {merged_est}'
record('PROB_CMS', 'CMS merge(1K+1K)',
       duration_ms=mt.duration_ms,
       extra={'merged_estimate_item0': merged_est, 'total': cms_merged.total})
gc.collect()

print(f'\nSuite 3 complete ✓')

## Suite 4: PROBABILISTIC_CRDT_PROPERTIES — CRDT Verification

Verify commutativity, associativity, and idempotency for all three probabilistic types.

In [ ]:
# ── Suite 4: Probabilistic CRDT Properties ────────────────────────────
from crdt_merge.verify import verify_commutative, verify_associative, verify_idempotent
from crdt_merge import probabilistic

# --- HLL ---
print('--- HLL CRDT properties ---')

def hll_gen_fn():
    hll = probabilistic.MergeableHLL(precision=14)
    n = random.randint(100, 1000)
    hll.add_all([f'item_{random.randint(0, 10000)}' for _ in range(n)])
    return hll

def hll_merge_fn(a, b):
    return a.merge(b)

def hll_eq_fn(a, b):
    return a.cardinality() == b.cardinality()

for prop_name, verify_fn in [('commutative', verify_commutative),
                              ('associative', verify_associative),
                              ('idempotent', verify_idempotent)]:
    with MemTracker() as mt:
        vr = verify_fn(hll_merge_fn, hll_gen_fn, trials=200, eq_fn=hll_eq_fn)
    record('PROB_CRDT', f'HLL {prop_name}',
           trials=vr.trials, duration_ms=mt.duration_ms,
           extra={'passed': vr.passed, 'failures': vr.failures})
    gc.collect()

# --- Bloom ---
print('\n--- Bloom CRDT properties ---')

def bloom_gen_fn():
    bl = probabilistic.MergeableBloom(capacity=1000, fp_rate=0.01)
    n = random.randint(50, 500)
    bl.add_all([f'item_{random.randint(0, 5000)}' for _ in range(n)])
    return bl

def bloom_merge_fn(a, b):
    return a.merge(b)

def bloom_eq_fn(a, b):
    # Check membership on a sample set
    test_items = [f'item_{i}' for i in range(100)]
    return all(a.contains(x) == b.contains(x) for x in test_items)

for prop_name, verify_fn in [('commutative', verify_commutative),
                              ('associative', verify_associative),
                              ('idempotent', verify_idempotent)]:
    with MemTracker() as mt:
        vr = verify_fn(bloom_merge_fn, bloom_gen_fn, trials=200, eq_fn=bloom_eq_fn)
    record('PROB_CRDT', f'Bloom {prop_name}',
           trials=vr.trials, duration_ms=mt.duration_ms,
           extra={'passed': vr.passed, 'failures': vr.failures})
    gc.collect()

# --- CMS ---
print('\n--- CMS CRDT properties ---')

def cms_gen_fn():
    cms = probabilistic.MergeableCMS(width=1000, depth=5)
    n = random.randint(50, 500)
    for _ in range(n):
        cms.add(f'item_{random.randint(0, 200)}', random.randint(1, 10))
    return cms

def cms_merge_fn(a, b):
    return a.merge(b)

def cms_eq_fn(a, b):
    test_items = [f'item_{i}' for i in range(50)]
    return all(a.estimate(x) == b.estimate(x) for x in test_items)

for prop_name, verify_fn in [('commutative', verify_commutative),
                              ('associative', verify_associative),
                              ('idempotent', verify_idempotent)]:
    with MemTracker() as mt:
        vr = verify_fn(cms_merge_fn, cms_gen_fn, trials=200, eq_fn=cms_eq_fn)
    record('PROB_CRDT', f'CMS {prop_name}',
           trials=vr.trials, duration_ms=mt.duration_ms,
           extra={'passed': vr.passed, 'failures': vr.failures})
    gc.collect()

print(f'\nSuite 4 complete ✓')

## Suite 5: SYSTEM_SANITY — Quick End-to-End Validation

Rapid smoke tests verifying backward-compatible features still work in v0.5.0.

In [ ]:
# ── Suite 5: System Sanity ────────────────────────────────────────────
from crdt_merge import (
    merge, merge_dicts, merge_with_provenance, export_provenance,
    merge_sorted_stream, StreamStats,
    GCounter, PNCounter, LWWRegister, ORSet, LWWMap,
    wire
)

checks = 0

# 1. Basic merge
df1 = pd.DataFrame({'id': [1,2,3], 'v': [10,20,30]})
df2 = pd.DataFrame({'id': [2,3,4], 'v': [25,35,40]})
merged = merge(df1, df2, key='id')
assert len(merged) == 4; checks += 1
print(f'  ✔ merge() — {len(merged)} rows')

# 2. merge_dicts
md = merge_dicts([{'id': 1, 'x': 10}], [{'id': 1, 'x': 20}], key='id')
assert len(md) == 1; checks += 1
print(f'  ✔ merge_dicts() — {len(md)} rows')

# 3. merge_with_provenance
result_df, prov = merge_with_provenance(df1, df2, key='id')
assert prov.total_rows > 0; checks += 1
prov_json = export_provenance(prov, format='json')
assert isinstance(json.loads(prov_json), dict); checks += 1
print(f'  ✔ merge_with_provenance() — {prov.total_rows} total rows')

# 4. merge_sorted_stream
def gen_sorted(n, off=0):
    for i in range(n):
        yield {'id': off + i, 'v': i}
stats = StreamStats()
rows = sum(len(b) for b in merge_sorted_stream(gen_sorted(100), gen_sorted(100, 50), key='id', stats=stats))
assert rows > 0; checks += 1
print(f'  ✔ merge_sorted_stream() — {rows} rows, {stats.batches_processed} batches')

# 5. GCounter
gc_obj = GCounter()
gc_obj.increment('a', 5)
gc_obj.increment('b', 3)
assert gc_obj.value == 8; checks += 1
print(f'  ✔ GCounter value={gc_obj.value}')

# 6. PNCounter
pn = PNCounter()
pn.increment('a', 10)
pn.decrement('a', 3)
assert pn.value == 7; checks += 1
print(f'  ✔ PNCounter value={pn.value}')

# 7. LWWRegister
reg = LWWRegister(value='hello')
reg.set('world', 'node1')
assert reg.value == 'world'; checks += 1
print(f'  ✔ LWWRegister value={reg.value}')

# 8. ORSet
os_obj = ORSet()
os_obj.add('alpha')
os_obj.add('beta')
assert os_obj.contains('alpha'); checks += 1
os_obj.remove('alpha')
assert not os_obj.contains('alpha'); checks += 1
print(f'  ✔ ORSet contains/remove works')

# 9. LWWMap
lm = LWWMap()
lm.set('key1', 'value1')
assert lm.get('key1') == 'value1'; checks += 1
assert lm.get('missing', 'default') == 'default'; checks += 1
print(f'  ✔ LWWMap get/set works')

# 10. Wire roundtrip
data = wire.serialize(gc_obj)
restored = wire.deserialize(data)
assert restored.value == gc_obj.value; checks += 1
print(f'  ✔ wire roundtrip GCounter')

# 11. Wire batch
batch_data = wire.serialize_batch([gc_obj, pn, reg])
restored_list = wire.deserialize_batch(batch_data)
assert len(restored_list) == 3; checks += 1
print(f'  ✔ wire batch roundtrip — {len(restored_list)} objects')

# 12. Probabilistic HLL
hll = probabilistic.MergeableHLL()
hll.add_all([f'x_{i}' for i in range(1000)])
assert hll.cardinality() > 0; checks += 1
print(f'  ✔ HLL cardinality={hll.cardinality()}')

record('SANITY', f'{checks} checks passed', extra={'checks': checks})
print(f'\nSuite 5 complete — {checks}/{checks} checks ✓')

## Results Summary

Aggregate all benchmark results.

In [ ]:
# ── Results Summary ────────────────────────────────────────────────────
print(f'Total results: {len(RESULTS)}')
print(f'{"Suite":<15s} {"Test":<45s} {"Duration (ms)":>14s} {"Mem (MB)":>10s}')
print('─' * 90)
for r in RESULTS:
    suite = r.get('suite', '')
    test  = r.get('test', '')
    ms    = f"{r['duration_ms']:,.1f}" if 'duration_ms' in r else '—'
    mb    = f"{r['mem_mb']:,.1f}" if 'mem_mb' in r else '—'
    print(f'{suite:<15s} {test:<45s} {ms:>14s} {mb:>10s}')

with open('v050_a100_results.json', 'w') as f:
    json.dump({'version': '0.5.0', 'results': RESULTS}, f, indent=2, default=str)
print('\n💾 Saved v050_a100_results.json')
print('\n🏁 All v0.5.0 A100 tests complete!')